In [ ]:
import os
import pydicom
import pandas as pd

In [8]:
# Configuration
# We step up from 'analysis' to the project folder, then into data
root_directory = os.path.join("..", "data", "PANCREAS_2", "PANCREAS_2")
output_filename = "dataset_audit_simple.csv"

# List to hold the data for every image we find
image_data_list = []

print(f"Scanning directory: {os.path.abspath(root_directory)}")

Scanning directory: /home/daniduhnev/projects/master-thesis/data/PANCREAS_2/PANCREAS_2


In [9]:
# Walk through all folders and subfolders
for current_folder, subfolders, files in os.walk(root_directory):
    
    for filename in files:
        # Ignore hidden files like .DS_Store
        if filename.startswith("."):
            continue
            
        full_path = os.path.join(current_folder, filename)
        
        try:
            # Open the DICOM file
            # stop_before_pixels=True makes this very fast as it skips the image data
            dicom_header = pydicom.dcmread(full_path, stop_before_pixels=True)
            
            # Extract the generic technical details
            # We use .get() to avoid crashing if a tag is missing
            rows = dicom_header.get("Rows", 0)
            columns = dicom_header.get("Columns", 0)
            channels = dicom_header.get("SamplesPerPixel", 1) # 1 is Grayscale, 3 is RGB
            modality = dicom_header.get("Modality", "Unknown")
            
            # Store the info
            file_info = {
                "filename": filename,
                "height": rows,
                "width": columns,
                "channels": channels,
                "modality": modality,
                "full_path": full_path
            }
            
            image_data_list.append(file_info)
            
        except Exception as error:
            print(f"Warning: Could not read file {filename}")

In [10]:
# Convert our list to a pandas DataFrame (like an Excel sheet)
df_images = pd.DataFrame(image_data_list)

In [12]:
# Generate Summary Report
print("Simple Dataset Audit Report\n")

# 1. Total Count
total_images = len(df_images)
print(f"Total Images Found: {total_images}")

# 2. Check Image Dimensions
# We want to know if all images are the same size
print("\nImage Resolutions (Height x Width):")
print(df_images.groupby(['height', 'width']).size())

# 3. Check Color Format
# This is critical: Do we have Color (3) or Grayscale (1)?
print("\nColor Channels (1=Grayscale, 3=RGB):")
print(df_images['channels'].value_counts())

# 4. Check Modality
# Confirming they are all Ultrasound (US)
print("\nModalities found:")
print(df_images['modality'].value_counts())

# Save the raw data to a CSV file for later
df_images.to_csv(output_filename, index=False)
print(f"\nAudit data saved to: {output_filename}")

Simple Dataset Audit Report

Total Images Found: 135

Image Resolutions (Height x Width):
height  width
768     1024     135
dtype: int64

Color Channels (1=Grayscale, 3=RGB):
channels
3    135
Name: count, dtype: int64

Modalities found:
modality
US    135
Name: count, dtype: int64

Audit data saved to: dataset_audit_simple.csv


In [14]:
import os

# Configuration
root_directory = os.path.join("..", "data", "PANCREAS_2", "PANCREAS_2")

print(f"Scanning for duplicates in: {os.path.abspath(root_directory)}")

duplicate_folders = []

# Walk through directories
for current_folder, subfolders, files in os.walk(root_directory):
    
    # Filter out hidden files (starts with .)
    valid_files = [f for f in files if not f.startswith(".")]
    
    # If a folder has more than 1 image, it's our "Duplicate"
    if len(valid_files) > 1:
        duplicate_folders.append({
            "folder": current_folder,
            "files": valid_files
        })

# Report findings
print("\n" + "-"*30)
print("DUPLICATE SCAN REPORT")
print("-"*(30))

if duplicate_folders:
    print(f"Found {len(duplicate_folders)} folder(s) containing multiple images:\n")
    
    for item in duplicate_folders:
        folder_name = os.path.basename(item['folder'])
        file_count = len(item['files'])
        print(f"📂 Study ID: {folder_name}")
        print(f"   Path:     {item['folder']}")
        print(f"   Count:    {file_count} images")
        print("   Files:")
        for f in item['files']:
            print(f"     - {f}")
else:
    print("✅ No duplicates found. Each folder contains exactly 1 image.")

Scanning for duplicates in: /home/daniduhnev/projects/master-thesis/data/PANCREAS_2/PANCREAS_2

------------------------------
DUPLICATE SCAN REPORT
------------------------------
Found 1 folder(s) containing multiple images:

📂 Study ID: 20191211
   Path:     ../data/PANCREAS_2/PANCREAS_2/48_02/20191211
   Count:    2 images
   Files:
     - 1.3.6.1.4.1.14519.5.2.1.9688.9989.278444632963972442685843638648
     - 1.3.6.1.4.1.14519.5.2.1.9688.9989.281574408988353255305175486060


In [15]:
import os
import pydicom
import numpy as np

# Path to the "Crime Scene" (The folder with 2 files)
duplicate_folder = os.path.join("..", "data", "PANCREAS_2", "PANCREAS_2", "48_02", "20191211")

# Get the two filenames
files = [f for f in os.listdir(duplicate_folder) if not f.startswith(".")]
files.sort()

print(f"🕵️‍♀️ Investigating: {duplicate_folder}")
print("-" * 60)

if len(files) < 2:
    print("Error: Expected 2 files, but found fewer. Did you already delete one?")
else:
    path1 = os.path.join(duplicate_folder, files[0])
    path2 = os.path.join(duplicate_folder, files[1])

    # Load them
    ds1 = pydicom.dcmread(path1)
    ds2 = pydicom.dcmread(path2)

    # 1. Compare Metadata
    print(f"{'Attribute':<20} | {'File 1':<35} | {'File 2':<35}")
    print("-" * 95)
    
    attributes = [
        ("Filename", files[0][-10:], files[1][-10:]), # Show last 10 chars
        ("Content Date", ds1.get("ContentDate", "N/A"), ds2.get("ContentDate", "N/A")),
        ("Content Time", ds1.get("ContentTime", "N/A"), ds2.get("ContentTime", "N/A")),
        ("Instance Number", ds1.get("InstanceNumber", "N/A"), ds2.get("InstanceNumber", "N/A")),
        ("Image Size", f"{ds1.Rows}x{ds1.Columns}", f"{ds2.Rows}x{ds2.Columns}"),
        ("File Size (Bytes)", os.path.getsize(path1), os.path.getsize(path2))
    ]

    for name, v1, v2 in attributes:
        print(f"{name:<20} | {str(v1):<35} | {str(v2):<35}")

    # 2. Compare Pixels
    print("-" * 95)
    if np.array_equal(ds1.pixel_array, ds2.pixel_array):
        print("✅ VISUAL VERDICT: The images are PIXEL-PERFECT IDENTICAL.")
        print("   -> Recommendation: Safe to delete one.")
    else:
        # Calculate difference
        diff = np.mean(np.abs(ds1.pixel_array.astype(float) - ds2.pixel_array.astype(float)))
        print(f"❌ VISUAL VERDICT: The images are DIFFERENT.")
        print(f"   -> Average Pixel Difference: {diff:.2f}")
        print("   -> Recommendation: Keep both. They are likely two different snapshots.")

🕵️‍♀️ Investigating: ../data/PANCREAS_2/PANCREAS_2/48_02/20191211
------------------------------------------------------------
Attribute            | File 1                              | File 2                             
-----------------------------------------------------------------------------------------------
Filename             | 5843638648                          | 5175486060                         
Content Date         | 20191211                            | 20191211                           
Content Time         | 133409.007000                       | 133409.007000                      
Instance Number      | 2                                   | 1                                  
Image Size           | 768x1024                            | 768x1024                           
File Size (Bytes)    | 2361612                             | 2361612                            
-----------------------------------------------------------------------------------------------
✅ 